# PrimeKV — Quick Test Notebook

Run these cells top-to-bottom to install, test, and interactively
compare KV cache strategies. Works on CPU (free tier) or GPU.

**From your phone:** just tap each cell and hit the play button.

## 1. Clone and install

In [ ]:
!rm -rf /content/PrimeKV
!git clone https://github.com/arunvenkatadri/PrimeKV.git
%cd /content/PrimeKV
!git checkout claude/scaffold-primekv-Q9QKN
!pip install -e ".[dev,web]" -q

## 2. Run unit tests (no network, no GPU, ~3 seconds)

In [ ]:
!pytest tests/ -v

## 3. Run the comparison CLI with real GPT-2

Downloads GPT-2 (124M) on first run (~500 MB). Takes 30-60s on CPU.

In [ ]:
!python benchmarks/compare.py --model gpt2 --decode-tokens 16 --max-length 64

## 4. Run comparison from Python (more control)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from primekv.eval import Workload, run_comparison
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier
from primekv.baselines import FullCache, H2OCache, UniformQuantCache

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

num_layers = model.config.n_layer

caches = {
    "full":        FullCache(num_layers),
    "uniform_int4": UniformQuantCache(num_layers, bits=4),
    "h2o":         H2OCache(num_layers, capacity=32),
    "primekv":     PrimeKVCache(
        num_layers=num_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 32},
    ),
}

workload = Workload(
    prompt="System: you are a helpful assistant. User: What is the capital of France? Assistant:",
    decode_tokens=16,
    max_length=64,
)

report = run_comparison(caches, workload, model, tok)
print(report.to_markdown())
print()
for r in report.results:
    if r.generated:
        print(f"--- {r.name} ---")
        print(r.generated)
        print()

## 5. Launch interactive Gradio UI

This creates a **public share link** you can open in any browser tab
(or send to a collaborator). The link is active as long as this cell
is running.

In [ ]:
%cd /content/PrimeKV
from webui.app import build_demo

demo = build_demo()
demo.launch(share=True)

## 6. Inspect PrimeKV tier distribution

In [ ]:
from primekv.metrics import tier_distribution, summarize_stats

# Use the primekv cache from step 4 (still in memory)
pkv = caches["primekv"]

print("Tier distribution:")
for tier, count in tier_distribution(pkv).items():
    print(f"  {tier:12s}  {count} tokens")

print("\nCache stats:")
for k, v in summarize_stats(pkv.stats).items():
    print(f"  {str(k):20s}  {v}")

## 7. Real-scale validation (GPU required)

This section runs the 2D eviction × quantization sweep on a **real instruction-tuned model** (Phi-2, 2.7B) with a long prompt (~1500 tokens). This is what goes into the paper.

**Before running this section:** go to `Runtime → Change runtime type → T4 GPU` (or A100 if you have Colab Pro). Then restart the runtime and run cell 1 again to reinstall.

Expected runtime: ~5-15 minutes on T4, ~2-5 minutes on A100.

In [ ]:
%cd /content/PrimeKV
import torch

# Verify GPU is available.
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
    "then restart the runtime and re-run cells 1 and 7.1."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 7.2 Load a real model

**Qwen2.5-3B-Instruct** is instruction-tuned, open-weight, first-class in transformers (no `trust_remote_code`), and doesn't require HuggingFace authentication. It's a genuine step up from GPT-2.

On an A100 it runs in FP16 with room to spare. If you're on a T4 and this is too big, downgrade to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
%cd /content/PrimeKV
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Qwen2.5-3B-Instruct is first-class in transformers (no trust_remote_code
# needed), instruction-tuned, open-weights (no HF auth), and ~6GB in FP16
# so it fits comfortably on an A100 or T4.
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",   # needed for standard past_key_values
)
model.eval()
print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Q heads: {model.config.num_attention_heads}")
print(f"KV heads: {getattr(model.config, 'num_key_value_heads', model.config.num_attention_heads)}")
print(f"Max context: {model.config.max_position_embeddings}")

### 7.3 A real long-context prompt

This is ~1500 tokens of Wikipedia-style content about the Eiffel Tower. It has the structure we want:
- Named entities early (Anchor tier should save them)
- Lots of factual elaboration (Supporting tier material)
- A continuation the model should complete fluently

In [ ]:
LONG_PROMPT = """The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889. Locally nicknamed La dame de fer, it was constructed as the centerpiece of the 1889 World's Fair and to crown the centennial anniversary of the French Revolution. Although initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world. The Eiffel Tower is the most-visited paid monument in the world; 6.91 million people ascended it in 2015.

The tower is 330 meters tall, about the same height as an 81-story building, and the tallest structure in Paris. Its base is square, measuring 125 meters on each side. During its construction, the Eiffel Tower surpassed the Washington Monument to become the tallest man-made structure in the world, a title it held for 41 years until the Chrysler Building in New York City was finished in 1930. It was the first structure to reach a height of 300 meters. Due to the addition of a broadcasting aerial at the top of the tower in 1957, it is now taller than the Chrysler Building by 5.2 meters.

The tower has three levels for visitors, with restaurants on the first and second levels. The top level's upper platform is 276 meters above the ground, the highest observation deck accessible to the public in the European Union. Tickets can be purchased to ascend by stairs or elevator to the first and second levels. The climb from ground level to the first level is over 300 steps, as is the climb from the first level to the second. Although there is a staircase to the top level, it is usually accessible only by elevator.

Eiffel openly acknowledged that inspiration for the tower came from the Latting Observatory built in New York City in 1853. In May 1884, working at home, Maurice Koechlin, a senior engineer at the Compagnie des Etablissements Eiffel, made a sketch of their idea, described by him as a great pylon, consisting of four lattice girders standing apart at the base and coming together at the top, joined together by metal trusses at regular intervals.

Gustave Eiffel initially showed little enthusiasm for the project, but he did approve further study, and the two engineers then asked Stephen Sauvestre, the head of the company's architectural department, to contribute to the design. Sauvestre added decorative arches to the base of the tower, a glass pavilion to the first level, and other embellishments. Eiffel bought the rights to the patent on 13 September 1884. By 30 March 1885, Eiffel presented his plans to the Société des Ingénieurs Civils; after discussing the technical problems and emphasising the practical uses of the tower, he finished his talk by saying the tower would symbolise not only the art of the modern engineer, but also the century of industry and science in which we are living.

The proposed tower had been a subject of controversy, drawing criticism from those who did not believe it was feasible and those who objected on artistic grounds. These objections were an expression of a long-standing debate in France about the relationship between architecture and engineering. It came to a head as work began at the Champ de Mars: a Committee of Three Hundred led by the prominent architect Charles Garnier and including some of the most important figures of the arts, such as Adolphe Bouguereau, Guy de Maupassant, Charles Gounod and Jules Massenet, sent a petition to Jean-Charles Alphand, the Minister of Works and Commissioner for the Exhibition, and it was published by Le Temps on 14 February 1887.

Some of the protests had such remarkable foresight that they would prove nearly prophetic. Gustave Eiffel himself later wrote that the tower would endure because it embodies"""

# Show the token count
tokens = tok(LONG_PROMPT, return_tensors="pt")
print(f"Prompt token count: {tokens.input_ids.shape[-1]}")

### 7.4 Run the 2D eviction × quantization sweep

This is the headline experiment. It runs ~27 configurations (1 full + 2 uniform + 4 h2o + 4 streamingllm + 4×3 primekv) against the same long prompt and produces the Pareto scatter plot.

**Expected runtime: 5-15 min on T4, 2-5 min on A100.**

In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_2d_tradeoff, plot_report
import logging
logging.basicConfig(level=logging.WARNING)

# Extended prompt — multi-topic Wikipedia content, ~3500 tokens.
# Long enough to stress the cache, short enough to run in ~10 min on A100.
EXTENDED_PROMPT = LONG_PROMPT + """

The transformer architecture, introduced by Vaswani et al. in 2017, revolutionized sequence modeling by replacing recurrent neural networks with self-attention mechanisms. Unlike RNNs, which process tokens sequentially and struggle with long-range dependencies due to vanishing gradients, transformers attend to all positions simultaneously. This parallelism enables efficient training on GPUs and has been the foundation of every major language model since, including GPT, BERT, T5, and Llama. The self-attention mechanism computes three projections of each input token — a query, a key, and a value — and uses the query-key similarity to determine how much each token should attend to every other token. This results in quadratic complexity in sequence length, which is the primary bottleneck for long-context inference.

The KV cache is a critical optimization in transformer inference. During autoregressive generation, the model produces one token at a time, with each new token requiring attention over all previous tokens. Rather than recomputing the keys and values for prior tokens at every decode step, the model caches these projections from prefill. This transforms decode from quadratic to linear complexity per token, at the cost of substantial memory consumption. The cache size scales with layers, heads, head dimension, and sequence length; for large models with long contexts, it often exceeds the model weights in memory footprint.

Several techniques have been proposed to reduce KV cache memory. Quantization methods like KIVI and KVQuant compress the cached tensors from FP16 to INT8 or INT4, trading precision for memory. Eviction methods like H2O and StreamingLLM drop tokens deemed less important based on cumulative attention scores or positional heuristics. Prompt caching stores cache entries across requests for shared prefixes. Each approach exploits a different dimension of redundancy. Combining them — as PrimeKV does with its tiered storage policies — can achieve higher compression ratios than any single technique alone.

Paris, the capital of France, has been a major European city since its founding in the 3rd century BC by a Celtic tribe called the Parisii. Located on the Seine River in north-central France, Paris is renowned for its art, fashion, gastronomy, and culture. The city is home to iconic landmarks including the Louvre Museum, Notre-Dame Cathedral, the Arc de Triomphe, and the Eiffel Tower. With a metropolitan population of 12 million, Paris remains one of the most populous urban regions in Europe.

Returning to the engineer who gave his name to the tower: Gustave Eiffel was born in Dijon, France in 1832. He studied at the École Centrale des Arts et Manufactures in Paris, graduating in 1855. Before the tower that bears his name, Eiffel designed the internal iron framework of the Statue of Liberty in 1885, demonstrating his mastery of large-scale metallic structures. The Eiffel Tower was initially intended as a temporary structure for the 1889 World's Fair, scheduled to be dismantled after twenty years. Its value as a radio transmission tower during World War I saved it from destruction, and it has remained standing ever since. Eiffel died in Paris in 1923 at the age of 91.

The influence of the Eiffel Tower on subsequent architecture and engineering cannot be overstated. It demonstrated that iron could be used to construct buildings of unprecedented height, foreshadowing the skyscraper era. Moreover, the tower's visual design — its lattice structure, curved base arches, and tapering silhouette — influenced countless buildings throughout the 20th century. What makes the tower particularly fascinating is that"""

prompt_tokens = tok(EXTENDED_PROMPT, return_tensors='pt').input_ids.shape[-1]
print(f"Extended prompt length: {prompt_tokens} tokens\n")

# Aggressive capacities relative to ~3500 tokens means 1-15% retention.
# This is the regime where eviction methods actually have to make hard choices.
report = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    eviction_caps=[32, 64, 128, 256, 512],
    precisions=["fp16", "int8", "int4"],
    decode_tokens=16,
    max_length=3500,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)

print(f"\nDone. {len(report.points)} configurations.")

### 7.5 Plot the results

In [ ]:
%cd /content/PrimeKV
import matplotlib.pyplot as plt

fig = plot_report(report, output_path="primekv_2d_sweep.png")
plt.show()

# Also save the raw numbers
with open("primekv_2d_sweep.csv", "w") as f:
    f.write(report.to_csv())
print("\nSaved: primekv_2d_sweep.png, primekv_2d_sweep.csv")

### 7.6 Headline numbers table

In [ ]:
%cd /content/PrimeKV
import pandas as pd

# Flatten the sweep results into a table.
rows = []
for p in report.points:
    rows.append({
        "cache": p.cache,
        "cap": p.extra.get("cap"),
        "precision": p.extra.get("precision"),
        "memory_MB": round(p.memory_bytes / (1024 * 1024), 2),
        "ratio": round(p.compression_ratio, 2),
        "ppl": round(p.perplexity, 3) if p.perplexity else None,
        "prefill_ms": round(p.prefill_ms, 1),
        "decode_ms": round(p.decode_ms, 1),
    })
df = pd.DataFrame(rows)
df = df.sort_values(["cache", "cap", "precision"])
print(df.to_string(index=False))

## 8. Honed sweeps: where does PrimeKV actually win?

The first round of results (Section 7) compared one-lever baselines (`uniform_int4`, pure `h2o`) against PrimeKV's two-lever design. That comparison is structurally unfair — int4 looked dominant on perplexity because no baseline was combining eviction *with* quantization. This section runs the sweeps that actually answer the question:

1. **8.1 Composed-baseline 2D sweep** — `h2o` and `streamingllm` now fill the full eviction × quantization plane via the new `H2OQuantCache` / `StreamingQuantCache` wrappers. PrimeKV no longer gets a free "only method with two levers" win.
2. **8.2 Long-context sweep** — capacity scales with prompt length (fixed *fraction* retained) so we test the regime where eviction has proportional leverage over fixed-ratio quantization.
3. **8.3 Seed aggregation** — same sweep with 5 sampling seeds, aggregated to mean/std. Without this, every number on a single chart is n=1.
4. **8.4 Reasoning-persistence harness** — perplexity averages across every token and hides whether the *right* tokens were kept. This runs small retention tests with a **filtered pass rate** (only count tests the `full` baseline passes), which was the bug in the earlier constraint-persistence charts.


### 8.1 2D sweep with composed baselines

Every capacity-driven method now fills the 2D surface. If PrimeKV still wins at the same compression ratio as `h2o_int4` / `streamingllm_int4`, that's real evidence the structural classifier is worth something. If not, we know where we are.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_2d_tradeoff, plot_report

report_2d = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    eviction_caps=[32, 64, 128, 256, 512],
    precisions=["fp16", "int8", "int4"],
    decode_tokens=16,
    max_length=3500,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)
print(f"\nDone. {len(report_2d.points)} configurations.")
fig = plot_report(report_2d, output_path="primekv_2d_composed.png")
print("Saved: primekv_2d_composed.png")


### 8.2 Long-context sweep (capacity scales with length)

Unlike Section 7's fixed-length test, this sweeps length from 512 → 16k tokens with **capacity = 10% × length** (min 32). That is the regime where eviction methods should gain leverage over uniform int4: most tokens in a 16k context genuinely don't matter, but uniform quantization compresses every token the same amount regardless.

If PrimeKV doesn't separate from `uniform_int4` here, it probably doesn't separate anywhere.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_long_context, DEFAULT_LONG_LENGTHS, plot_report

# Qwen2.5-3B supports 32k context; trim the top of the default grid if
# your session has memory pressure.
long_lengths = [512, 1024, 2048, 4096, 8192]

report_lc = sweep_long_context(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    lengths=long_lengths,
    capacity_fraction=0.1,
    min_capacity=32,
    precisions=["int4"],
    caches=["uniform_int4", "h2o_int4", "streamingllm_int4", "primekv"],
    decode_tokens=16,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)
print(f"\nDone. {len(report_lc.points)} configurations.")
fig = plot_report(report_lc, output_path="primekv_long_context.png")
print("Saved: primekv_long_context.png")


### 8.3 Seed aggregation

With argmax decoding, a sweep is deterministic and n=1 is the same as n=100. To get honest error bars we switch to top-k sampling (`sample_top_k=50`) and run the same sweep across 5 seeds. `aggregate_reports` collapses the runs into mean/std per cell, so every chart plotted from it has real variance on it.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_pareto, aggregate_reports

seeds = [0, 1, 2, 3, 4]
reports = []
for s in seeds:
    r = sweep_pareto(
        model=model,
        tokenizer=tok,
        prompt=EXTENDED_PROMPT[:2000],  # shorter prompt keeps wall-time reasonable
        capacities=[32, 64, 128, 256],
        caches=["full", "uniform_int4", "h2o_int4", "streamingllm_int4", "primekv"],
        decode_tokens=16,
        max_length=1500,
        device="cuda",
        seed=s,
        sample_top_k=50,
        sample_temperature=1.0,
        progress=lambda msg, s=s: print(f"  seed={s} {msg}"),
    )
    reports.append(r)

agg = aggregate_reports(reports)
print(f"\nAggregated {len(reports)} runs into {len(agg.points)} cells.")
print("Each point now has perplexity_std, memory_bytes_std etc. in `extra`:")
for p in agg.points[:3]:
    extras = {k: v for k, v in p.extra.items() if k.endswith('_std') or k == 'n_runs'}
    print(f"  {p.cache} cap={p.sweep_value}  ppl={p.perplexity:.3f}  {extras}")


### 8.4 Reasoning-persistence harness (filtered pass rate)

The earlier constraint-persistence chart had `primekv_spacy` passing 100% of tests while `full` passed only 50% — which is a physical impossibility for cache quality. The cause: the denominator included tests where `full` was already failing for non-cache reasons (model couldn't comply with the constraint, couldn't chain the facts), so any perturbation of generation could flip those tests and look like a win.

`run_reasoning_suite` + `filtered_pass_rate` fix this: only count tests where `full` itself passes. That's the honest cache-quality metric — among questions the uncompressed model can actually answer, how many survive under each cache?

The built-in suite is a small smoke test. Swap in LongBench / RULER for a real evaluation.


In [ ]:
%cd /content/PrimeKV
from primekv.reasoning import (
    default_reasoning_suite,
    run_reasoning_suite,
    plot_reasoning_report,
)
from primekv.baselines import (
    FullCache,
    H2OQuantCache,
    StreamingQuantCache,
    UniformQuantCache,
)
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier

n_layers = model.config.num_hidden_layers

# Factories so each (test, seed, cache) run gets a fresh cache — matters
# for PrimeKV because reset() preserves the classifier assignment.
factories = {
    "full":               lambda: FullCache(n_layers),
    "uniform_int4":       lambda: UniformQuantCache(n_layers, bits=4),
    "h2o_int4":           lambda: H2OQuantCache(n_layers, capacity=64, bits=4),
    "streamingllm_int4":  lambda: StreamingQuantCache(n_layers, num_sinks=4, window=64, bits=4),
    "primekv":            lambda: PrimeKVCache(
        n_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=8, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 64},
    ),
}
caches = {name: f() for name, f in factories.items()}

tests = default_reasoning_suite()
seeds = [0, 1, 2]  # tiny: sampling noise, not a full statistical test

report_rs = run_reasoning_suite(
    caches=caches,
    tests=tests,
    model=model,
    tokenizer=tok,
    seeds=seeds,
    device="cuda",
    sample_top_k=40,
    sample_temperature=0.8,
    cache_factories=factories,
    progress=lambda msg: print(f"  {msg}"),
)

print("\nSummary:")
for cache_name, row in report_rs.summary().items():
    print(
        f"  {cache_name:20s}  filtered={row['filtered_pass_rate']:.2%}  "
        f"raw={row['raw_pass_rate']:.2%}  "
        f"(n_valid_tests={row['n_valid_tests']})"
    )

fig = plot_reasoning_report(report_rs, output_path="primekv_reasoning.png")
print("\nSaved: primekv_reasoning.png")


### 8.5 What to look for in the plots

Across Sections 8.1–8.4, three specific questions are worth checking:

- **2D plot (8.1):** does the PrimeKV cloud have any point that strictly dominates the `h2o_int4` / `streamingllm_int4` points at equal compression ratio? That's the real PrimeKV-wins-on-quality signal. If all the composed-baseline points sit on or below PrimeKV's frontier, the structural classifier isn't buying anything that eviction+quant alone doesn't already buy.
- **Long-context (8.2):** at 8k/16k tokens, does PrimeKV pull away from `uniform_int4`? Uniform quantization is a fixed 4× compression regardless of length; eviction-at-fixed-fraction compresses proportionally to length, so the two should separate as context grows. If they don't, the structural claim is weak.
- **Reasoning (8.4):** on the *filtered* pass rate (not the raw), does PrimeKV beat `h2o_int4`? This is where structure *should* matter — tests that require recalling specific tokens punish eviction that drops those tokens, regardless of how precisely the survivors are stored.

Any one of these three being positive is a real wedge. All three being flat means the method doesn't have a regime, and we should know that now rather than later.


## 9. Diagnostics: is chunked PrimeKV well-founded for this model?

Before building a `ChunkedPrimeKVCache`, three measurements answer whether the design has any chance of working. Each is one model forward pass — cheap relative to a sweep — and they answer specific yes/no questions:

- **9.1** Does this model have lost-in-the-middle / attention sinks? *Without a U-shape, chunked compression of the middle has no license to be lossy.*
- **9.2** Does the rule classifier's notion of "important" agree with the model's notion of "attended-to"? *If anchors cluster where attention does, the structure is coherent; if not, we have a contradiction.*
- **9.3** Can the model do prefill while attending over a compressed past? *This is the gating result for streaming-chunked PrimeKV (design A from the conversation). If quality collapses when prefill attends over int4'd history, streaming is dead and we fall back to post-hoc chunking only.*

Run these after Section 7 has loaded `model`, `tok`, and `EXTENDED_PROMPT`. They produce three small plots and a printed table; nothing is committed to the sweep harness yet.


### 9.1 Attention received per position

For each source position *k*, the average attention weight given to *k* by every query position that could attend to it (i.e., positions ≥ *k* under the causal mask), averaged across heads and layers. Plotted on a log scale to make the sink spike and the middle valley both visible.


In [ ]:
%cd /content/PrimeKV
import torch
import numpy as np
import matplotlib.pyplot as plt

# Cap the diagnostic length so output_attentions stays in memory on a T4.
DIAG_LEN = 1024
ids = tok(EXTENDED_PROMPT, return_tensors="pt", truncation=True, max_length=DIAG_LEN).input_ids.to(model.device)
seq_len = int(ids.shape[-1])

with torch.no_grad():
    out = model(input_ids=ids, output_attentions=True, use_cache=False)

# attentions: tuple over layers, each (batch=1, heads, seq, seq), softmax-normalized.
# For each key position k, average over (heads, layers, queries q with q >= k).
n_layers = len(out.attentions)
attn_per_pos = torch.zeros(seq_len, dtype=torch.float32, device=ids.device)
for layer_attn in out.attentions:
    a = layer_attn[0].float().mean(dim=0)  # avg over heads → (seq, seq)
    # column k: sum of attention from queries i = k..seq_len-1 (others are zero by causal mask)
    col_sums = a.sum(dim=0)
    counts = torch.arange(seq_len, 0, -1, device=ids.device, dtype=torch.float32)  # seq_len, ..., 1
    attn_per_pos += col_sums / counts
attn_per_pos = (attn_per_pos / n_layers).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(attn_per_pos, color="#1f77b4")
ax.set_xlabel("source position")
ax.set_ylabel("avg attention received (per querier)")
ax.set_title(f"Attention-by-position curve ({MODEL_NAME}, n={seq_len})")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("primekv_attn_by_position.png", dpi=120)
plt.show()

# Quick numeric summary so we know if the U-shape exists at all.
sink_band = attn_per_pos[:16].mean()
middle_band = attn_per_pos[seq_len//4:3*seq_len//4].mean()
recent_band = attn_per_pos[-64:].mean()
print(f"\nattention-received summary:")
print(f"  sink  (first 16 tokens):       {sink_band:.4e}")
print(f"  middle (25%-75% of context):   {middle_band:.4e}")
print(f"  recent (last 64 tokens):       {recent_band:.4e}")
print(f"  sink/middle ratio:   {sink_band/middle_band:.1f}x")
print(f"  recent/middle ratio: {recent_band/middle_band:.1f}x")
print()
print("Reading: ratios > ~3 mean a clear U-shape and license for aggressive middle compression.")
print("Ratios near 1 mean attention is uniform and chunked compression has no leverage here.")


### 9.2 Classifier-vs-attention agreement

Same prompt, this time we run PrimeKV's rule classifier and overlay tier assignments against the attention curve from 9.1. The story we *want* to see: anchors cluster where attention is highest (start), supporting / filler tokens dominate the middle valley. If the classifier instead spreads semantic tokens uniformly through the middle, the rule classifier and the model are saying different things about importance — and we'd need to fix that before chunking helps.


In [ ]:
%cd /content/PrimeKV
from primekv.classifier import RuleBasedClassifier, Tier
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

clf = RuleBasedClassifier(anchor_prefix_len=16, semantic_stride=3)
assignment = clf.classify(input_ids=ids[0])
tiers = assignment.tiers.cpu().numpy()

# Tier counts for context.
tier_names = {0: "ANCHOR", 1: "SEMANTIC", 2: "SUPPORTING", 3: "FILLER"}
tier_colors = {0: "#cc3333", 1: "#ff7f0e", 2: "#3377aa", 3: "#999999"}

print("Tier counts:")
for t, name in tier_names.items():
    n = int((tiers == t).sum())
    print(f"  {name:11s} {n:4d}  ({100*n/len(tiers):.1f}%)")

# Plot: attention curve on top, tier band underneath.
fig, (ax_attn, ax_tiers) = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                                        gridspec_kw={"height_ratios": [4, 1]})
ax_attn.plot(attn_per_pos, color="#1f77b4")
ax_attn.set_yscale("log")
ax_attn.set_ylabel("avg attention received")
ax_attn.set_title(f"Classifier vs attention agreement ({MODEL_NAME})")
ax_attn.grid(True, alpha=0.3)

# Tier band as a strip of colored rectangles.
for pos in range(seq_len):
    ax_tiers.axvspan(pos - 0.5, pos + 0.5, color=tier_colors[int(tiers[pos])], alpha=0.85)
ax_tiers.set_yticks([])
ax_tiers.set_xlabel("source position")
ax_tiers.set_ylabel("tier")

handles = [mpatches.Patch(color=tier_colors[t], label=tier_names[t]) for t in tier_names]
ax_tiers.legend(handles=handles, loc="upper right", ncol=4, fontsize=8)
fig.tight_layout()
fig.savefig("primekv_classifier_vs_attention.png", dpi=120)
plt.show()

# Quantitative agreement: average attention received per tier.
print("\nAverage attention received, per tier:")
for t, name in tier_names.items():
    mask = (tiers == t)
    if mask.any():
        mean_attn = float(attn_per_pos[mask].mean())
        print(f"  {name:11s}  mean_attn={mean_attn:.4e}")
print()
print("Reading: ANCHOR should have the highest mean attention, FILLER the lowest.")
print("If they're not ordered correctly, the rule classifier and the model disagree on importance.")


### 9.3 Can the model do prefill over a compressed past?

This is the gating experiment for **streaming-chunked PrimeKV**. We split the prompt in half, compress the first half's K/V, then run prefill on the second half *attending over the lossy first half*. We measure perplexity on the second-half tokens. If int4 compression of the prefix barely shifts ppl, streaming is viable. If it explodes, streaming is dead — and the chunked design has to fall back to post-hoc reorganization (which still works but doesn't reduce peak memory).


In [ ]:
%cd /content/PrimeKV
import math
import torch
from primekv.adapters.gpt2 import _extract_kv_pairs, reconstruct_past_kv, _wrap_as_hf_cache
from primekv.baselines import FullCache, UniformQuantCache, H2OQuantCache, StreamingQuantCache

# Use a longer slice for this — prefill error compounds with prefix length, so
# a too-short prefix understates the real cost.
ids_full = tok(EXTENDED_PROMPT, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
total_len = int(ids_full.shape[-1])
split = total_len // 2
prefix_ids = ids_full[:, :split]
suffix_ids = ids_full[:, split:]

n_layers = model.config.num_hidden_layers
prefix_len = int(prefix_ids.shape[-1])
print(f"prefix_len={prefix_len}, suffix_len={int(suffix_ids.shape[-1])}\n")


def suffix_ppl_with_prefix_cache(prefix_cache, name: str) -> float:
    """Run prefix prefill into prefix_cache, reconstruct lossy past, run
    suffix prefill against that lossy past with labels, return perplexity."""
    prefix_cache.reset()
    with torch.no_grad():
        out_pre = model(input_ids=prefix_ids, use_cache=True)
        kv_pairs = _extract_kv_pairs(out_pre.past_key_values)
        for layer, (k, v) in enumerate(kv_pairs):
            for pos in range(prefix_len):
                prefix_cache.put(layer, pos, k[0, :, pos, :], v[0, :, pos, :])
        lossy = reconstruct_past_kv(prefix_cache, kv_pairs, prefix_len)
        lossy_past = _wrap_as_hf_cache(list(lossy))
        out_suf = model(
            input_ids=suffix_ids,
            past_key_values=lossy_past,
            use_cache=False,
            labels=suffix_ids,
        )
    return float(math.exp(out_suf.loss.item()))


# Strict memory budget for the H2O / Streaming variants: keep ~25% of the prefix.
keep = max(32, prefix_len // 4)

variants = {
    "full_prefix":          FullCache(n_layers),
    "int4_prefix":          UniformQuantCache(n_layers, bits=4),
    "int8_prefix":          UniformQuantCache(n_layers, bits=8),
    "h2o_int4_prefix":      H2OQuantCache(n_layers, capacity=keep, bits=4),
    "streaming_int4_prefix": StreamingQuantCache(n_layers, num_sinks=4, window=keep, bits=4),
}

results = {}
for name, c in variants.items():
    print(f"running {name} ...")
    results[name] = suffix_ppl_with_prefix_cache(c, name)

baseline = results["full_prefix"]
print()
print(f"{'variant':25s}  {'suffix ppl':>10s}  {'Δ vs full':>10s}")
print("-" * 55)
for name, ppl in results.items():
    print(f"{name:25s}  {ppl:>10.3f}  {ppl - baseline:>+10.3f}")
print()
print("Reading:")
print("  Δ < 0.05  → streaming compression is viable for this variant.")
print("  Δ in 0.05-0.20 → marginal; would need careful tuning.")
print("  Δ > 0.20 → streaming over this kind of compressed past is too lossy.")
print()
print("If int4_prefix shows tiny Δ but h2o_int4_prefix shows large Δ, the issue")
print("is eviction during prefill, not quantization. That points at design (A')")
print("with quantization-only streaming compression and post-hoc eviction.")


### 9.4 What the three answers imply

After running 9.1–9.3, the design space collapses to one of these regimes:

| 9.1 (U-shape) | 9.2 (classifier matches) | 9.3 (lossy prefill survives) | What to build |
|---|---|---|---|
| Strong | Yes | Yes | **Streaming three-zone PrimeKV** (anchor + recent FP16 + compressed middle). The full design from earlier — real peak memory wedge over int4. |
| Strong | Yes | No | Post-hoc chunked PrimeKV with U-shape-aware reduce. No peak memory savings but tier policies that exploit the U-shape; still a candidate to beat int4 on quality at fixed memory. |
| Strong | **No** | — | Fix the classifier first. The rule classifier disagreeing with the model's attention pattern is a real bug; chunking is downstream of fixing it. |
| Weak (flat attention) | — | — | Don't build chunked PrimeKV. Without a U-shape there's no leverage; the existing global PrimeKV is the right object. |

Whichever row we're in, the answer is concrete and we can move on.


## 10. New mechanisms: three-zone, streaming-chunked, trained classifier

Section 8 ran the existing PrimeKV on a fair surface; Section 9 measured whether chunked PrimeKV was even well-founded. This section actually *tries* three new mechanisms and compares them on the existing harness:

1. **ThreeZoneCache** (10.1) — permanent anchor + rolling FP16 window + structurally-compressed middle. Direct application of the U-shape insight from 9.1.
2. **Streaming-chunked prefill** (10.2) — `streaming_run_with_cache` processes the prompt in chunks; each chunk's prefill attends only to the (lossy) reconstruction of prior chunks. This is the design that gives a real peak-memory wedge int4 fundamentally can't replicate.
3. **Trained MLP classifier** (10.3) — `train_mlp_classifier_on_prompt` fits the MLPClassifier to weak labels derived from teacher attention. Tests whether attention-supervised classification beats the positional rule classifier.

These cells are deliberately *try-it-and-see*: each prints a few headline numbers. The expensive comparison sweeps come in the existing sections — these are scaffolding to confirm the mechanisms work end-to-end and produce believable numbers on a real model before any deeper investment.


### 10.1 ThreeZoneCache vs the composed baselines

Compare three-zone against `full`, `uniform_int4`, and `h2o_int4` on the same prompt. Three-zone should sit close to `uniform_int4` on quality and memory — but the *shape* of its loss should be different (it preserves anchor + recent FP16 exactly).


In [ ]:
%cd /content/PrimeKV
from primekv.baselines import FullCache, UniformQuantCache, H2OQuantCache, ThreeZoneCache
from primekv.eval import Workload, run_comparison

n_layers = model.config.num_hidden_layers
prompt_len = tok(EXTENDED_PROMPT, return_tensors='pt').input_ids.shape[-1]
print(f"prompt length: {prompt_len} tokens\n")

# Three-zone with a realistic capacity for a ~3500-token prompt.
zone_window = 256          # last 256 non-anchor tokens stay FP16
zone_anchor = 16
caches = {
    "full":              FullCache(n_layers),
    "uniform_int4":      UniformQuantCache(n_layers, bits=4),
    "h2o_int4":          H2OQuantCache(n_layers, capacity=zone_window, bits=4),
    "three_zone_int4":   ThreeZoneCache(
        n_layers, num_anchor=zone_anchor, recent_window=zone_window, middle_bits=4
    ),
}

rep = run_comparison(
    caches,
    Workload(prompt=EXTENDED_PROMPT, decode_tokens=16, max_length=3500),
    model, tok, device="cuda",
)
print(rep.to_markdown())

# Show the populated-zone breakdown for the three-zone cache.
tz = caches["three_zone_int4"]
print("\nthree-zone populated counts:")
print(f"  anchor: {sum(len(p) for p in tz._anchor)} positions  (FP16, pinned)")
print(f"  recent: {sum(len(p) for p in tz._recent)} positions  (FP16, rolling window={zone_window})")
print(f"  middle: {sum(len(p) for p in tz._middle)} positions  (int4)")


### 10.2 Streaming-chunked prefill

Run the same caches through `streaming_run_with_cache` with chunk_size=512. The interesting comparison is *not* perplexity vs the global prefill (the streaming variant has cumulative error by design) but whether the perplexity is *acceptable* and whether the inference-time peak memory profile differs.

If `three_zone_int4`'s streaming ppl is within ~0.1 of its global ppl, streaming-chunked PrimeKV is viable on this model. If it blows up, fall back to post-hoc chunking.


In [ ]:
%cd /content/PrimeKV
from primekv.adapters.gpt2 import streaming_run_with_cache
from primekv.baselines import FullCache, UniformQuantCache, ThreeZoneCache
from primekv.eval import Workload

chunk_size = 512
wl = Workload(prompt=EXTENDED_PROMPT, decode_tokens=16, max_length=3500)

streaming_rows = []
for name, cache in [
    ("full",            FullCache(n_layers)),
    ("uniform_int4",    UniformQuantCache(n_layers, bits=4)),
    ("three_zone_int4", ThreeZoneCache(
        n_layers, num_anchor=zone_anchor, recent_window=zone_window, middle_bits=4
    )),
]:
    print(f"streaming {name} ...")
    r = streaming_run_with_cache(name, cache, model, tok, wl, chunk_size=chunk_size, device="cuda")
    streaming_rows.append((name, r))
    print(f"  ppl={r.perplexity:.3f}  mem={r.memory_bytes/1024/1024:.2f}MB  "
          f"prefill_ms={r.prefill_ms:.0f}")

print()
print("Compare with global-prefill numbers from cell 10.1 to see how much")
print("perplexity drifts under chunk-by-chunk prefill over lossy past.")


### 10.3 Trained MLP classifier

Fit `MLPClassifier` to teacher-attention labels on the prompt itself, then plug it into a PrimeKVCache and compare against the rule classifier.

**Scope caveat:** this is an in-prompt fit, not a corpus-level pretrain. Useful for testing the *mechanism* — does attention-supervised classification differ meaningfully from the positional rule — but the resulting classifier won't generalize to other prompts. A real classifier needs a separate training pipeline. If this in-prompt scaffold already shows the rule baseline losing badly to the trained variant, that's a strong signal the rule classifier was the bottleneck.


In [ ]:
%cd /content/PrimeKV
import torch
from primekv.classifier import RuleBasedClassifier, MLPClassifier, Tier
from primekv.classifier_training import train_mlp_classifier_on_prompt
from primekv.cache import PrimeKVCache
from primekv.baselines import FullCache
from primekv.eval import Workload, run_comparison

# Train the MLP classifier on the prompt itself.
ids = tok(EXTENDED_PROMPT, return_tensors='pt', truncation=True, max_length=3500).input_ids.to(model.device)
d_model = model.config.hidden_size
mlp_clf = MLPClassifier(d_model=d_model, hidden=128).to(model.device)
print(f"training MLP classifier ({mlp_clf.net[0].in_features} -> 128 -> 4) on {ids.shape[-1]} positions ...")
train_log = train_mlp_classifier_on_prompt(
    mlp_clf, model, ids,
    epochs=80, lr=1e-3, anchor_prefix_len=16, hidden_layer_index=2, verbose=False,
)
print(f"  final loss: {train_log['final_loss']:.4f}")
print(f"  final accuracy vs teacher labels: {train_log['final_accuracy']:.1%}")
print(f"  per-tier accuracy: {train_log['per_tier_accuracy']}")

# Compare rule vs trained classifier on the harness.
mlp_clf.eval()
support_cap = 256
caches = {
    "full":              FullCache(n_layers),
    "primekv_rule":      PrimeKVCache(
        n_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=16, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: support_cap},
    ),
    "primekv_mlp":       PrimeKVCache(
        n_layers,
        classifier=mlp_clf,
        max_entries_per_tier={Tier.SUPPORTING: support_cap},
    ),
}
rep = run_comparison(
    caches,
    Workload(prompt=EXTENDED_PROMPT, decode_tokens=16, max_length=3500),
    model, tok, device="cuda",
)
print()
print(rep.to_markdown())
print()
print("Reading: if primekv_mlp beats primekv_rule by more than ~0.05 ppl at")
print("similar compression, the rule classifier was the bottleneck. If they're")
print("within noise, the classifier isn't what's holding PrimeKV back.")


### 10.4 What this section settles

Across 10.1–10.3 you've actually tried three new mechanisms on the real model. The cells print enough to read off the answer to each:

- **Three-zone wins on quality at equal memory?** → the U-shape design is the right shape for this model.
- **Streaming-chunked perplexity acceptable?** → streaming prefill over a lossy past is viable; chunked PrimeKV gets a real peak-memory wedge.
- **Trained classifier beats the rule classifier?** → the rule classifier was the bottleneck and the spaCy/rule comparison from Section 7 was missing the right baseline all along.

Any one of these landing positively is real new evidence; all three landing flat means the structural-compression direction needs a bigger move than these mechanisms provide.
